# Theoretical-player scoring playground

Edit the `player` dict a few cells down and re-run to see how the six trained models value your hypothetical player.

**Prerequisite** (run once from the project root):
```bash
python src/models_v2.py
```
This trains + saves all six `(model_type, feature_set)` bundles into `./models/`.

**Required fields:** `position` (GK/DF/MF/FW), `age_at_transfer`, `minutes`.

**Recommended fields** (fall back to 0 / training-median otherwise): `goals`, `assists`, `expected_goals`, `expected_assisted_goals`, `non_penalty_xg`, `key_passes`, `progressive_passes`, `tackles_won`, `interceptions`, `clearances`, `clean_sheets`, `saves`, `height_in_cm`, `foot`, `league`, `nationality_group`, `sub_position`, `contract_years_remaining`, `from_club`, `to_club`, `transfer_year`.

## 1. Bootstrap imports

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
from predict import predict_all_models, derive_features, available_models
from club_lookup import list_clubs, get_club_level, resolve_club_name

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

print("Trained bundles on disk:", available_models())

Trained bundles on disk: [('linear', 'traditional'), ('random_forest', 'traditional'), ('xgboost', 'traditional'), ('linear', 'with_modern'), ('random_forest', 'with_modern'), ('xgboost', 'with_modern')]


## 2. Browse clubs to pick realistic from/to pair

In [3]:
# Biggest clubs by all-time median MV
list_clubs(limit=15)

,club_id,club_name,n_years,alltime_level_log,alltime_median_eur
0,418,Real Madrid Club de Fútbol,21,16.30,"12,000,000.00"
1,131,Futbol Club Barcelona,22,16.30,"12,000,000.00"
2,985,Manchester United Football Club,22,16.21,"11,000,000.00"
3,583,Paris Saint-Germain Football Club,22,16.12,"10,000,000.00"
4,31,Liverpool Football Club,22,16.12,"10,000,000.00"
5,281,Manchester City Football Club,22,16.12,"10,000,000.00"
6,148,Tottenham Hotspur Football Club,22,15.89,"8,000,000.00"
7,11,Arsenal Football Club,22,15.76,"7,000,000.00"
8,703,Nottingham Forest Football Club,19,15.52,"5,500,000.00"
9,631,Chelsea Football Club,22,15.42,"5,000,000.00"


In [4]:
# Search by name substring
list_clubs(search="madrid", limit=10)

,club_id,club_name,n_years,alltime_level_log,alltime_median_eur
0,418,Real Madrid Club de Fútbol,21,16.30,"12,000,000.00"
1,13,Club Atlético de Madrid S.A.D.,22,15.20,"4,000,000.00"
2,367,Rayo Vallecano de Madrid S.A.D.,22,13.82,"1,000,000.00"


In [7]:
# Sanity-check how fuzzy matching resolves informal names
for q in ["manchester u", "real madrid", "brighton"]:
    print(f"  {q!r:20s} -> {resolve_club_name(q)!r}  "
          f"(level={get_club_level(q, year=2024)})")

  'manchester u'       -> 'Manchester United Football Club'  (level=17.281246492014)
  'real madrid'        -> 'Real Madrid Club de Fútbol'  (level=16.811242881518265)
  'brighton'           -> 'Brighton and Hove Albion Football Club'  (level=16.705882371415992)


## 3. Define your theoretical player and score

Re-run this cell after editing the dict.

In [8]:
player = {
    # --- required ---
    "position": "FW",             # GK, DF, MF, or FW
    "age_at_transfer": 22,
    "minutes": 2800,

    # --- attacking ---
    "goals": 19,
    "assists": 7,
    "expected_goals": 16.8,
    "expected_assisted_goals": 5.9,
    "non_penalty_xg": 14.2,
    "key_passes": 38,
    "progressive_passes": 110,

    # --- defending / GK (leave as 0 for most outfield players) ---
    "tackles_won": 10,
    "interceptions": 5,
    "clearances": 3,
    "clean_sheets": 0,
    "saves": 0,

    # --- profile ---
    "height_in_cm": 182,
    "foot": "right",
    "league": "Premier League",
    "nationality_group": "England",
    "sub_position": "Centre-Forward",
    "contract_years_remaining": 3.5,

    # --- transfer context (drives from_club_level / to_club_level) ---
    "from_club": "Brighton",
    "to_club": "Manchester City",
    "transfer_year": 2024,
}

predictions = predict_all_models(player)
predictions

,Model,Feature Set,Predicted MV (EUR),Log Pred,Test R²
0,linear,traditional,"74,333,781.03",18.12,0.57
1,random_forest,traditional,"32,752,382.53",17.30,0.65
2,xgboost,traditional,"36,514,548.63",17.41,0.69
3,linear,with_modern,"100,151,557.34",18.42,0.59
4,random_forest,with_modern,"39,078,907.67",17.48,0.66
5,xgboost,with_modern,"42,053,808.11",17.55,0.72


## 4. Compare: does adding modern stats move the prediction?

In [9]:
pivot = predictions.pivot_table(
    index="Model",
    columns="Feature Set",
    values="Predicted MV (EUR)",
    aggfunc="first",
)
pivot["modern_delta_eur"] = pivot.get("with_modern", 0) - pivot.get("traditional", 0)
pivot

Feature Set,traditional,with_modern,modern_delta_eur
Model,,,
linear,"74,333,781.03","100,151,557.34","25,817,776.31"
random_forest,"32,752,382.53","39,078,907.67","6,326,525.14"
xgboost,"36,514,548.63","42,053,808.11","5,539,259.48"


## 5. Inspect the derived feature row (debugging aid)

Useful if a prediction looks suspicious — check what features actually got computed.

In [10]:
row = derive_features(player)
print(f"Derived feature row: {row.shape[1]} columns")
row.T.head(40)

Derived feature row: 76 columns


,0
age_at_transfer,22.00
age_squared,484.00
height_in_cm,182
contract_years_remaining,3.50
position,FW
sub_position,Centre-Forward
foot,right
league,Premier League
nationality_group,England
from_club_level,16.71


## 6. Scenario sweep: vary one input and watch the prediction move

How much does age alone move predicted value for this forward?

In [11]:
scenarios = []
for age in [18, 21, 24, 27, 30, 33]:
    variant = {**player, "age_at_transfer": age}
    preds = predict_all_models(variant)
    row = {"age": age}
    for _, r in preds.iterrows():
        key = f"{r['Model']}__{r['Feature Set']}"
        row[key] = r["Predicted MV (EUR)"]
    scenarios.append(row)

sweep = pd.DataFrame(scenarios).set_index("age")
sweep

,linear__traditional,random_forest__traditional,xgboost__traditional,linear__with_modern,random_forest__with_modern,xgboost__with_modern
age,,,,,,
18,"59,859,946.75","31,847,488.11","31,496,185.87","81,826,193.24","37,319,400.07","27,733,962.38"
21,"72,176,377.36","32,717,021.18","33,381,455.98","97,578,271.11","38,925,628.33","37,379,895.85"
24,"75,045,706.34","31,811,089.76","37,429,194.21","100,460,047.33","38,771,800.88","41,228,144.70"
27,"67,286,607.74","28,243,885.51","33,125,406.94","89,292,118.19","34,970,190.70","42,527,212.84"
30,"52,023,963.10","21,416,064.73","23,724,196.41","68,519,212.01","28,644,439.01","31,480,090.09"
33,"34,685,682.61","14,040,053.52","12,798,121.47","45,393,229.39","21,032,440.16","16,225,499.22"
